# 08 — Sixteen Ways to Measure the Same Disagreement

Notebooks 04, 05 and 07 all compare pruned models with **four** distributional
distances: KLD, JSD, EMD and Chamfer. That choice was never justified against
alternatives, and the four are not independent — three of them read the same
per-position probability vectors, and two are f-divergences of the same pair.

This notebook widens the field to **sixteen** measures (the original four plus
twelve added in `pruning_metrics.metrics.distributions`) and asks three
questions of them:

1. **Do they agree?** If all sixteen induce the same geometry over the 13 model
   variants, the choice of metric is arbitrary and nothing downstream depends
   on it. If they split into families, the choice is a real modelling decision.
2. **Do any of them behave badly on this data?** Top-5 truncation and the
   −50 logprob fill are not neutral; some divergences are much more exposed to
   them than others.
3. **Which one best predicts real degradation?** Distance from the unpruned
   baseline is regressed on measured pass@1 drop, exactly as in notebook 05,
   but now for all sixteen.

Everything here runs off the local `results/tf_cache` — **no AWS, no GPU**. The
one-pass batch build takes a few minutes on 14 cores; after that the matrices
are cached and the notebook re-runs in seconds.

**Scope.** The v1 13-variant set (3 calibration sources × 4 pruning levels +
shared baseline, across 3 eval benchmarks), which is the set that has measured
`pass@1` numbers attached to it. The v2 232-variant set in notebook 07 keeps
the original four metrics; extending it would be another full matrix build.


In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env", override=False)
except ImportError:
    pass

print("REPO_ROOT =", REPO_ROOT)


## Configuration

Deliberately no `boto3` import: unlike notebooks 04 and 07 this one never
touches S3. It reads `results/tf_cache`, which notebook 04 populates.


In [ ]:
import concurrent.futures
import csv
import json
import re
import time

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from pruning_metrics.metrics import METRIC_INFO, METRIC_NAMES, compute_all

NOTEBOOK_DIR = Path.cwd()
RESULTS_DIR = NOTEBOOK_DIR / "results"
TF_CACHE_DIR = RESULTS_DIR / "tf_cache"
FIGURES_DIR = RESULTS_DIR / "metric_family_figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["arc_challenge", "gsm8k", "humaneval"]
NONZERO_LEVELS = [20, 40, 60, 80]
BENCHMARK_LABELS = {
    "gsm8k": "GSM8K",
    "humaneval": "HumanEval+",
    "arc_challenge": "ARC-Challenge",
}
CAL_COLORS = {
    "arc_challenge": "tab:blue",
    "gsm8k": "tab:orange",
    "humaneval": "tab:green",
}

# Row 0 is the shared unpruned baseline; level=0 is calibration-independent.
ROW_META = [{"cal": "baseline", "level": 0}] + [
    {"cal": cal, "level": lv} for cal in DATASETS for lv in NONZERO_LEVELS
]
N_MODELS = len(ROW_META)

# The four that notebooks 04/05/07 already use. Their matrices are on disk and
# are the provenance of every committed figure, so this notebook loads them and
# never rewrites them.
LEGACY_METRICS = ("kld", "jsd", "emd", "chamfer")

N_WORKERS = max(1, (os.cpu_count() or 4) - 2)

print(f"{N_MODELS} model variants x {len(DATASETS)} benchmarks x "
      f"{len(METRIC_NAMES)} metrics")
print(f"workers: {N_WORKERS}   figures -> {FIGURES_DIR}")


## The sixteen measures

They fall into four families, distinguished by *what they are handed*, which
matters more than the formula:

| family | representation | consequence |
|---|---|---|
| **f-divergence** | the two probability vectors at one position, aligned onto the union of their top-5 supports | sensitive to token identity; a token in one model's top-5 and not the other's is a large disagreement |
| **geometry** | the same two aligned vectors, treated as plain points in ℝⁿ | no information-theoretic content — the control condition for whether the divergences earn their keep |
| **transport** | each model's own logprob values as atom positions on the line | *blind to token identity* — measures whether the shape of the confidence profile moved, not what it moved onto |
| **point-cloud** | the whole sequence at once, matched nearest-neighbour rather than by index | ignores alignment; asks whether the two models visit the same region of prediction space at all |

The transport/f-divergence split is the sharpest one: a pruned model that
predicts a completely different token with exactly the same confidence scores
**maximal** total variation and **zero** Wasserstein.

Three relationships within the list are worth stating before the numbers,
because they are consequences of the definitions and not findings:

- `jeffreys` = `kld` + `rkld`, exactly.
- `renyi05` = 2 · `bhattacharyya`, exactly — the Rényi-½ divergence is a
  *linear* rescaling of Bhattacharyya, and linearity survives summing over
  positions. It is kept as a wiring check: any agreement analysis that does not
  report a correlation of exactly 1.000 between those two columns is broken.
- `renyi2` = ln(1 + `chisq`) **per position only**. That map is nonlinear, so
  it does *not* survive summation, and the two are genuinely different measures
  of a 13-model matrix. The empirical check is in the agreement section.


In [ ]:
_fam_order = {"f-divergence": 0, "geometry": 1, "transport": 2, "point-cloud": 3}
hdr = f"{'key':<14s} {'label':<16s} {'family':<13s} {'sym':<4s} {'bound':<7s} formula"
print(hdr)
print("-" * len(hdr))
for name in sorted(METRIC_NAMES, key=lambda n: (_fam_order[METRIC_INFO[n].family],
                                                METRIC_NAMES.index(n))):
    info = METRIC_INFO[name]
    bound = "-" if info.bounded is None else f"{info.bounded:.3f}"
    mark = "*" if name in LEGACY_METRICS else " "
    print(f"{name + mark:<14s} {info.label:<16s} {info.family:<13s} "
          f"{'yes' if info.symmetric else 'no':<4s} {bound:<7s} {info.formula}")
print("\n* = already used by notebooks 04/05/07")
print("bound = maximum contribution of a single token position; '-' = unbounded")


## Index the per-token cache

Same scan as notebook 05 cell 7: `{combo: {task_id: {level: path}}}` over the
nine calibration × evaluation combinations. Takes ~35 s for 37,845 files.


In [ ]:
def scan_combo_cache(combo_key: str) -> dict:
    """Return {task_id: {level: path_to_per_token.json}} for one TF run."""
    base = TF_CACHE_DIR / combo_key
    index: dict[str, dict[int, Path]] = {}
    if not base.exists():
        print(f"  WARNING: {base} not found - run notebook 04 first.")
        return index
    for level_dir in sorted(base.glob("level=*")):
        m = re.match(r"level=(\d+)", level_dir.name)
        if not m:
            continue
        level = int(m.group(1))
        for sample_dir in sorted(level_dir.iterdir()):
            pt_path = sample_dir / "per_token.json"
            if not pt_path.exists():
                continue
            with pt_path.open() as fh:
                meta = json.load(fh)
            tid = meta.get("task_id", sample_dir.name)
            index.setdefault(tid, {})[level] = pt_path
    return index


_t0 = time.perf_counter()
TF_INDEX: dict[str, dict] = {}
for _cal in DATASETS:
    for _ev in DATASETS:
        _key = f"{_cal}_{_ev}"
        TF_INDEX[_key] = scan_combo_cache(_key)
        print(f"  {_key:34s} {len(TF_INDEX[_key]):5d} tasks")
print(f"indexed in {time.perf_counter() - _t0:.1f}s")


## Pairwise distance matrices — one pass, sixteen metrics

Each metric gives a 13×13 matrix whose (i, j) entry is the mean over eval
examples of that distance between model *i* and model *j*'s per-token outputs.

The cost is dominated by JSON parsing and by rebuilding the union-support
vectors, both of which are shared across metrics — so
`compute_all` produces all sixteen for roughly the price of one, where the
original per-metric loop in notebook 05 paid for each separately.

Two rules this cell follows:

- **Never rewrite `pairwise_dist_{bench}_{kld,jsd,emd,chamfer}.npy`.** Those
  four are the provenance of the figures in notebooks 04/05/07. They are loaded
  from disk; only the twelve new files are written. Dropping `chamfer` from the
  batch is also the single biggest saving, since it compares every position
  against every other and costs more than the other fifteen combined.
- **Only the upper triangle is computed**, then mirrored. For the two
  directional measures (`kld`, `chisq` — and `rkld`, `renyi2`) that means the
  stored matrix is the *i < j* direction symmetrised by construction, not the
  true asymmetric distance. `rkld` is therefore genuinely new data rather than
  `kld`'s transpose: it is `KL(model_j ‖ model_i)` where `kld` is
  `KL(model_i ‖ model_j)`, both stored in symmetric form.


In [ ]:
# Module-level worker state. Fork start method (Linux default) lets the pool
# inherit these without pickling; ProcessPoolExecutor.map only ships task ids.
_W_BENCH = ""
_W_METRICS: list[str] = []


def _task_contribution(tid: str):
    """All pairwise distances for one eval example, summed into a (M, 13, 13) block."""
    tokens: dict[int, list] = {}
    for idx, meta in enumerate(ROW_META):
        cal = DATASETS[0] if meta["level"] == 0 else meta["cal"]
        by_level = TF_INDEX.get(f"{cal}_{_W_BENCH}", {}).get(tid, {})
        path = by_level.get(meta["level"])
        if path is not None:
            with path.open() as fh:
                tokens[idx] = json.load(fh).get("per_token", [])

    acc = np.zeros((len(_W_METRICS), N_MODELS, N_MODELS), dtype=np.float64)
    counts = np.zeros((N_MODELS, N_MODELS), dtype=np.int64)
    for i in range(N_MODELS):
        ti = tokens.get(i, [])
        if not ti:
            continue
        for j in range(i + 1, N_MODELS):
            tj = tokens.get(j, [])
            if not tj:
                continue
            values = compute_all(ti, tj, metrics=_W_METRICS)
            vec = np.array([values[m] for m in _W_METRICS], dtype=np.float64)
            finite = np.isfinite(vec)
            acc[finite, i, j] += vec[finite]
            acc[finite, j, i] += vec[finite]
            counts[i, j] += 1
            counts[j, i] += 1
    return acc, counts


def _cache_path(eval_bench: str, metric: str) -> Path:
    return RESULTS_DIR / f"pairwise_dist_{eval_bench}_{metric}.npy"


def build_all_matrices(eval_bench: str) -> dict[str, np.ndarray]:
    """Load cached metrics, batch-compute the rest, return all sixteen."""
    global _W_BENCH, _W_METRICS

    matrices: dict[str, np.ndarray] = {}
    missing: list[str] = []
    for metric in METRIC_NAMES:
        path = _cache_path(eval_bench, metric)
        if path.exists():
            matrices[metric] = np.load(path)
        else:
            missing.append(metric)

    if not missing:
        print(f"  {eval_bench}: all {len(METRIC_NAMES)} matrices cached")
        return matrices

    combo_keys = [f"{c}_{eval_bench}" for c in DATASETS
                  if f"{c}_{eval_bench}" in TF_INDEX]
    task_ids = sorted(set.intersection(*[set(TF_INDEX[k]) for k in combo_keys]))
    print(f"  {eval_bench}: {len(task_ids)} tasks x "
          f"{N_MODELS * (N_MODELS - 1) // 2} pairs, computing {len(missing)}: "
          f"{', '.join(missing)}")

    _W_BENCH, _W_METRICS = eval_bench, missing
    acc = np.zeros((len(missing), N_MODELS, N_MODELS), dtype=np.float64)
    counts = np.zeros((N_MODELS, N_MODELS), dtype=np.int64)
    started = time.perf_counter()
    with concurrent.futures.ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        for block, block_counts in pool.map(_task_contribution, task_ids, chunksize=8):
            acc += block
            counts += block_counts

    denom = np.maximum(counts, 1)
    for idx, metric in enumerate(missing):
        matrices[metric] = np.where(counts > 0, acc[idx] / denom, 0.0)
        np.save(_cache_path(eval_bench, metric), matrices[metric])
    print(f"  {eval_bench}: built in {time.perf_counter() - started:.1f}s")
    return matrices


MATRICES: dict[str, dict[str, np.ndarray]] = {}
_t0 = time.perf_counter()
for _bench in DATASETS:
    MATRICES[_bench] = build_all_matrices(_bench)
print(f"\ntotal {time.perf_counter() - _t0:.1f}s")


## Provenance check

The batch path must reproduce the legacy per-metric path exactly, or the four
cached matrices this notebook just loaded would not be comparable with the
twelve it just computed. HumanEval+ is the cheap benchmark (33 tasks), so it is
rebuilt from scratch here and compared bit-for-bit against what is on disk.

A `max|Δ| = 0` result is not luck: `compute_all` and the individual functions
share the same per-position kernels and apply them in the same order, so the
two routes produce identical floats by construction. This cell is the
regression test for that guarantee against files written months earlier.


In [ ]:
_W_BENCH, _W_METRICS = "humaneval", list(LEGACY_METRICS)
_keys = [f"{c}_humaneval" for c in DATASETS if f"{c}_humaneval" in TF_INDEX]
_tasks = sorted(set.intersection(*[set(TF_INDEX[k]) for k in _keys]))

_acc = np.zeros((len(LEGACY_METRICS), N_MODELS, N_MODELS))
_counts = np.zeros((N_MODELS, N_MODELS), dtype=np.int64)
with concurrent.futures.ProcessPoolExecutor(max_workers=N_WORKERS) as _pool:
    for _block, _bc in _pool.map(_task_contribution, _tasks, chunksize=4):
        _acc += _block
        _counts += _bc

print(f"{'metric':<10s} {'max|delta|':>12s}   verdict")
_all_exact = True
for _i, _m in enumerate(LEGACY_METRICS):
    _rebuilt = np.where(_counts > 0, _acc[_i] / np.maximum(_counts, 1), 0.0)
    _delta = float(np.abs(_rebuilt - MATRICES["humaneval"][_m]).max())
    _all_exact &= _delta == 0.0
    print(f"{_m:<10s} {_delta:>12.3e}   "
          f"{'identical' if _delta == 0.0 else 'DIFFERS - investigate'}")
assert _all_exact, "batch build diverged from the cached per-metric matrices"
print("\nBatch and per-metric builds agree bit-for-bit on all four legacy metrics.")


## Scale audit — before comparing anything

These sixteen numbers are not on a common scale, and two of them are not on a
*usable* scale. Before any correlation or regression, look at what each measure
actually returns on a real level-0 vs level-80 pair, and at how much of its
value comes from positions where the two models' top-5 sets do not overlap at
all — the regime where the −50 logprob fill, not the model, sets the value.


In [ ]:
# How often do the two models' top-5 sets not overlap at all? This is the
# regime where the -50 logprob fill, rather than the model, sets a divergence's
# value - so it is the number that decides whether chi-squared is usable.
# Sampled over the first PROBE_TASKS tasks of every calibration combo rather
# than a single example, since the rate varies several-fold between tasks.
PROBE_TASKS = 60

_disjoint_rows = []
for _bench in DATASETS:
    _pos = _dis = 0
    for _cal in DATASETS:
        _combo = f"{_cal}_{_bench}"
        for _tid in sorted(TF_INDEX.get(_combo, {}))[:PROBE_TASKS]:
            _levels = TF_INDEX[_combo][_tid]
            if 0 not in _levels or 80 not in _levels:
                continue
            _a = json.load(_levels[0].open())["per_token"]
            _b = json.load(_levels[80].open())["per_token"]
            for _sa, _sb in zip(_a, _b):
                _ids_a = {t["token_id"] for t in _sa.get("top_alternatives", [])}
                _ids_b = {t["token_id"] for t in _sb.get("top_alternatives", [])}
                if not _ids_a or not _ids_b:
                    continue
                _pos += 1
                _dis += not (_ids_a & _ids_b)
    _disjoint_rows.append((_bench, _dis, _pos))

print(f"Top-5 supports that do not overlap at all, level 0 vs level 80 "
      f"(first {PROBE_TASKS} tasks per calibration combo):")
for _bench, _dis, _pos in _disjoint_rows:
    print(f"  {BENCHMARK_LABELS[_bench]:<15s} {_dis:6d}/{_pos:6d} positions "
          f"= {100 * _dis / max(_pos, 1):5.2f}%")
_dis_all = sum(r[1] for r in _disjoint_rows)
_pos_all = sum(r[2] for r in _disjoint_rows)
DISJOINT_RATE = 100 * _dis_all / max(_pos_all, 1)
print(f"  {'pooled':<15s} {_dis_all:6d}/{_pos_all:6d} positions = "
      f"{DISJOINT_RATE:5.2f}%\n")

# Per-metric cost of one such position, versus the observed matrix scale.
_disjoint_cost = {}
for _name in METRIC_NAMES:
    if _name in ("emd", "wasserstein2", "chamfer"):
        _disjoint_cost[_name] = float("nan")   # not union-support measures
        continue
    _a = [{"position": 0, "top_alternatives": [
        {"token_id": i, "token_text": "", "logprob": lp}
        for i, lp in enumerate([0.0, -6.0, -7.0, -8.0, -9.0])]}]
    _b = [{"position": 0, "top_alternatives": [
        {"token_id": 10 + i, "token_text": "", "logprob": lp}
        for i, lp in enumerate([0.0, -6.0, -7.0, -8.0, -9.0])]}]
    _disjoint_cost[_name] = compute_all(_a, _b, metrics=[_name])[_name]

_rows = []
hdr = (f"{'metric':<14s} {'family':<13s} {'max D (gsm8k)':>14s} "
       f"{'cost/disjoint pos':>18s} {'ratio':>9s}")
print(hdr)
print("-" * len(hdr))
for _name in METRIC_NAMES:
    _dmax = float(MATRICES["gsm8k"][_name].max())
    _cost = _disjoint_cost[_name]
    _ratio = _dmax / _cost if _cost and np.isfinite(_cost) and _cost > 0 else float("nan")
    _rows.append(dict(metric=_name, family=METRIC_INFO[_name].family,
                      max_distance=_dmax, disjoint_position_cost=_cost,
                      positions_equivalent=_ratio))
    print(f"{_name:<14s} {METRIC_INFO[_name].family:<13s} {_dmax:>14.4g} "
          f"{_cost:>18.4g} {_ratio:>9.4g}")

_SCALE_CSV = RESULTS_DIR / "metric_scale_audit.csv"
with _SCALE_CSV.open("w", newline="") as _fh:
    _w = csv.DictWriter(_fh, fieldnames=list(_rows[0]))
    _w.writeheader()
    _w.writerows(_rows)
print(f"\n-> {_SCALE_CSV}")
print("'ratio' reads as: the whole matrix maximum is worth this many "
      "disjoint-support positions.")


### How Euclidean is each metric space?

Half the reducers used in notebooks 05 and 07 (PCA, LLE) cannot consume a
distance matrix directly — they need coordinates, which come from classical MDS
(Torgerson double-centering). That step keeps only the positive eigenvalues of
the centered matrix, so whatever mass sits in the negative ones is *silently
discarded*. `neg_ratio` is the fraction discarded, and it is a property of the
metric, not of the reducer: a metric with a high `neg_ratio` is one where the
PCA and LLE maps are drawings of something other than the matrix that was
handed in.

Hellinger is the interesting case to watch — it is a Euclidean distance between
square-root embeddings by construction, so it should come out near zero.


In [ ]:
from pruning_metrics.embedding import mds_spectrum

_spec_rows = []
hdr = f"{'metric':<14s} " + " ".join(f"{BENCHMARK_LABELS[b]:>14s}" for b in DATASETS)
print(hdr)
print("-" * len(hdr))
for _name in METRIC_NAMES:
    _cells = []
    for _bench in DATASETS:
        _s = mds_spectrum(MATRICES[_bench][_name])
        _spec_rows.append(dict(eval_bench=_bench, metric=_name,
                               **{k: _s[k] for k in ("n_pos", "neg_ratio", "var_2d")}))
        _cells.append(f"{_s['neg_ratio']:>14.4f}")
    print(f"{_name:<14s} " + " ".join(_cells))

_SPEC_CSV = RESULTS_DIR / "metric_mds_spectrum.csv"
with _SPEC_CSV.open("w", newline="") as _fh:
    _w = csv.DictWriter(_fh, fieldnames=list(_spec_rows[0]))
    _w.writeheader()
    _w.writerows(_spec_rows)
print(f"\nneg_ratio = |negative eigenvalue mass| / positive mass, discarded by "
      f"classical MDS.\n-> {_SPEC_CSV}")


## Do the sixteen measures agree?

For each pair of metrics, the Spearman correlation of their 78 upper-triangle
entries — the same quantity a Mantel test reports, computed on all
16 × 15 / 2 = 120 metric pairs per benchmark.

Spearman rather than Pearson deliberately: the metrics differ by twelve orders
of magnitude in scale (see the audit above), and the question is whether they
*order* model pairs the same way, not whether they agree on units. It also
means χ², whose linear structure is dominated by the ε guard, still gets a fair
hearing here.


In [ ]:
from scipy.stats import spearmanr

from pruning_metrics.metrics import mantel

_iu = np.triu_indices(N_MODELS, k=1)


def agreement_matrix(bench: str) -> np.ndarray:
    """Spearman rho between the upper triangles of every pair of metrics."""
    vectors = np.array([MATRICES[bench][m][_iu] for m in METRIC_NAMES])
    rho = np.ones((len(METRIC_NAMES), len(METRIC_NAMES)))
    for i in range(len(METRIC_NAMES)):
        for j in range(i + 1, len(METRIC_NAMES)):
            r = spearmanr(vectors[i], vectors[j]).statistic
            rho[i, j] = rho[j, i] = r
    return rho


AGREEMENT = {b: agreement_matrix(b) for b in DATASETS}

_i05, _ibh = METRIC_NAMES.index("renyi05"), METRIC_NAMES.index("bhattacharyya")
_i2, _ichi = METRIC_NAMES.index("renyi2"), METRIC_NAMES.index("chisq")
print("Wiring checks (see the definitions section):")
for _b in DATASETS:
    print(f"  {BENCHMARK_LABELS[_b]:<14s} rho(renyi05, bhattacharyya) = "
          f"{AGREEMENT[_b][_i05, _ibh]:.6f}   "
          f"rho(renyi2, chisq) = {AGREEMENT[_b][_i2, _ichi]:.6f}")
assert all(abs(AGREEMENT[b][_i05, _ibh] - 1.0) < 1e-9 for b in DATASETS), \
    "renyi05 must be a linear rescaling of bhattacharyya"

_rows = []
for _b in DATASETS:
    for _i, _a in enumerate(METRIC_NAMES):
        for _j, _c in enumerate(METRIC_NAMES):
            if _i < _j:
                _rows.append(dict(eval_bench=_b, metric_a=_a, metric_b=_c,
                                  spearman=AGREEMENT[_b][_i, _j]))
_AGREE_CSV = RESULTS_DIR / "metric_agreement.csv"
with _AGREE_CSV.open("w", newline="") as _fh:
    _w = csv.DictWriter(_fh, fieldnames=["eval_bench", "metric_a", "metric_b", "spearman"])
    _w.writeheader()
    _w.writerows(_rows)
print(f"\n{len(_rows)} metric pairs -> {_AGREE_CSV}")

_vals = np.array([AGREEMENT["gsm8k"][i, j]
                  for i in range(len(METRIC_NAMES))
                  for j in range(i + 1, len(METRIC_NAMES))])
print(f"GSM8K pairwise agreement: median rho = {np.median(_vals):.3f}, "
      f"min = {_vals.min():.3f}, {(_vals > 0.95).sum()}/{_vals.size} pairs above 0.95")

# The 78 upper-triangle entries are not independent observations - they share
# models - so significance needs a Mantel permutation test rather than a
# Spearman p-value. Run against JSD, the metric the rest of the repo uses.
print("\nMantel test against JSD (999 row/column permutations):")
_j = METRIC_NAMES.index("jsd")
hdr = f"{'metric':<14s} " + " ".join(f"{BENCHMARK_LABELS[b]:>22s}" for b in DATASETS)
print(hdr)
print("-" * len(hdr))
for _name in METRIC_NAMES:
    if _name == "jsd":
        continue
    _cells = []
    for _b in DATASETS:
        _r, _p = mantel(MATRICES[_b]["jsd"], MATRICES[_b][_name],
                        permutations=999, method="spearman", seed=0)
        _cells.append(f"r={_r:+.3f} p={_p:.3f}".rjust(22))
    print(f"{_name:<14s} " + " ".join(_cells))


In [ ]:
# Every observed value sits in [0.83, 1.00]. On a full [0, 1] scale the whole
# grid renders one flat colour, so the scale starts at 0.80 and the caption
# carries the fact that nothing falls below it.
_VMIN = 0.80
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for ax, bench in zip(axes, DATASETS):
    rho = AGREEMENT[bench]
    im = ax.imshow(rho, cmap="RdYlBu_r", vmin=_VMIN, vmax=1.0)
    ax.set_xticks(range(len(METRIC_NAMES)))
    ax.set_yticks(range(len(METRIC_NAMES)))
    ax.set_xticklabels(METRIC_NAMES, rotation=90, fontsize=7)
    ax.set_yticklabels(METRIC_NAMES, fontsize=7)
    ax.set_title(BENCHMARK_LABELS[bench], fontsize=11, fontweight="bold")
    for i in range(len(METRIC_NAMES)):
        for j in range(len(METRIC_NAMES)):
            ax.text(j, i, f"{rho[i, j]:.2f}".lstrip("0"),
                    ha="center", va="center", fontsize=4.6,
                    color="white" if rho[i, j] < 0.885 else "black")
fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02, label="Spearman rho")
_lo = min(AGREEMENT[b].min() for b in DATASETS)
fig.suptitle("Agreement between distance measures — rank correlation of the 78 "
             "model-pair distances\n"
             f"1.00 = identical ordering of every model pair. Colour scale starts "
             f"at {_VMIN:.2f}; the lowest value anywhere here is {_lo:.3f}.",
             fontsize=13, fontweight="bold", y=1.02)
_out = FIGURES_DIR / "metric_agreement_heatmap.png"
fig.savefig(_out, dpi=140, bbox_inches="tight")
print(f"Saved: {_out}")
plt.show()


### Which families do they actually fall into?

The heatmap above is read more easily as a tree. Metrics are clustered by
1 − ρ (average linkage) on the pooled agreement across the three benchmarks, so
the grouping is the *empirical* family structure — as opposed to the taxonomy
by representation stated at the top, which was a claim about the formulas.
Where the two disagree, the empirical one is the finding.


In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

_pooled = np.mean([AGREEMENT[b] for b in DATASETS], axis=0)
_dist = np.clip(1.0 - _pooled, 0.0, 2.0)
np.fill_diagonal(_dist, 0.0)
_dist = 0.5 * (_dist + _dist.T)
_link = linkage(squareform(_dist, checks=False), method="average")

_FAMILY_COLORS = {"f-divergence": "tab:blue", "geometry": "tab:red",
                  "transport": "tab:green", "point-cloud": "tab:purple"}

# The whole tree fits under 1 - rho = 0.07, so the threshold has to sit inside
# that range or every branch comes out one colour.
_CUT = 0.012
fig, ax = plt.subplots(figsize=(13, 6.5))
_dn = dendrogram(_link, labels=list(METRIC_NAMES), ax=ax,
                 color_threshold=_CUT, above_threshold_color="0.55",
                 leaf_rotation=38)
ax.set_ylabel("1 - Spearman rho  (average over the three benchmarks)", fontsize=9)
ax.axhline(_CUT, color="0.7", lw=0.8, ls=":", zorder=0)
ax.set_title("Empirical families of distributional distance\n"
             "label colour = family by construction; tree = family by behaviour",
             fontsize=12, fontweight="bold")
for _lbl in ax.get_xmajorticklabels():
    _lbl.set_color(_FAMILY_COLORS[METRIC_INFO[_lbl.get_text()].family])
    _lbl.set_fontsize(9)
    _lbl.set_ha("right")
ax.legend(handles=[mpatches.Patch(color=c, label=f) for f, c in _FAMILY_COLORS.items()],
          fontsize=8, loc="upper right")
fig.tight_layout()
_out = FIGURES_DIR / "metric_dendrogram.png"
fig.savefig(_out, dpi=150, bbox_inches="tight")
print(f"Saved: {_out}")
plt.show()

print(f"Whole tree height (worst disagreement between any two groups): "
      f"{_link[-1, 2]:.4f} in 1 - rho.\n")
print(f"Clusters at 1 - rho < {_CUT} (measures that are near-interchangeable here):")
from scipy.cluster.hierarchy import fcluster
_flat = fcluster(_link, t=_CUT, criterion="distance")
for _cid in sorted(set(_flat)):
    _members = [m for m, c in zip(METRIC_NAMES, _flat) if c == _cid]
    if len(_members) > 1:
        print(f"  {', '.join(_members)}")


## R² — which distance actually predicts degradation?

The test that matters. For each (benchmark, metric): regress measured `pass@1`
drop on the model's distance from the unpruned baseline — row 0 of the distance
matrix — over the 13 variants. `raw` uses the distance directly; `pca` and
`isomap` use the radius in the 2-D embedding, which is the ceiling those
reducers are trying to preserve and the quantity notebook 05 reports.

PCA and Isomap only. Notebook 07's quality scoring found t-SNE, UMAP and LLE
preserve local neighbourhoods but destroy the distance scale, so their radius
is not a meaningful regressor; adding them here would only reproduce that
result sixteen times.


In [ ]:
from pruning_metrics.embedding import embed_2d
from pruning_metrics.metrics import baseline_distances, linear_r2

with open(RESULTS_DIR / "metric_space_combined.csv") as _fh:
    _combined = list(csv.DictReader(_fh))
DROP = {(r["cal"], r["eval"], int(float(r["pruning_level"]))): float(r["pass_at_1_drop"])
        for r in _combined}

REDUCERS_USED = ["pca", "isomap"]
EMBEDDINGS: dict[tuple[str, str, str], np.ndarray] = {}

r2_rows = []
_t0 = time.perf_counter()
for _bench in DATASETS:
    # Row 0 is the shared baseline, whose drop is 0 by definition.
    _y = np.array([0.0 if m["cal"] == "baseline"
                   else DROP.get((m["cal"], _bench, int(m["level"])), np.nan)
                   for m in ROW_META], dtype=float)
    for _name in METRIC_NAMES:
        _D = MATRICES[_bench][_name]
        _fit = linear_r2(_D[0], _y)
        r2_rows.append(dict(eval_bench=_bench, metric=_name, reducer="raw",
                            **{k: _fit[k] for k in ("n", "r2", "r", "slope")}))
        for _red in REDUCERS_USED:
            _coords, _ = embed_2d(_D, _red, random_state=42)
            EMBEDDINGS[(_bench, _name, _red)] = _coords
            _fit = linear_r2(baseline_distances(_coords, 0), _y)
            r2_rows.append(dict(eval_bench=_bench, metric=_name, reducer=_red,
                                **{k: _fit[k] for k in ("n", "r2", "r", "slope")}))

_R2_CSV = RESULTS_DIR / "metric_family_r2.csv"
with _R2_CSV.open("w", newline="") as _fh:
    _w = csv.DictWriter(_fh, fieldnames=["eval_bench", "metric", "reducer",
                                         "n", "r2", "r", "slope"])
    _w.writeheader()
    _w.writerows(r2_rows)
print(f"{len(r2_rows)} rows in {time.perf_counter() - _t0:.1f}s -> {_R2_CSV}\n")


def _r2(bench, metric, reducer):
    for row in r2_rows:
        if (row["eval_bench"], row["metric"], row["reducer"]) == (bench, metric, reducer):
            return row["r2"]
    return float("nan")


hdr = f"{'metric':<14s} " + " ".join(
    f"{BENCHMARK_LABELS[b][:9] + '/' + r:>17s}" for b in DATASETS for r in ("raw",))
print("R^2 of pass@1 drop on raw distance from baseline")
print(hdr)
print("-" * len(hdr))
_best = {}
for _name in METRIC_NAMES:
    _vals = [_r2(b, _name, "raw") for b in DATASETS]
    _best[_name] = np.nanmean(_vals)
    print(f"{_name:<14s} " + " ".join(f"{v:>17.3f}" for v in _vals))
print("\nRanked by mean raw R^2 across the three benchmarks:")
for _rank, (_name, _score) in enumerate(
        sorted(_best.items(), key=lambda kv: -kv[1]), start=1):
    _mark = "  <- notebooks 04/05/07 use this" if _name in LEGACY_METRICS else ""
    print(f"  {_rank:2d}. {_name:<14s} {_score:.3f}{_mark}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 9.5), sharey=True)
_order = sorted(METRIC_NAMES, key=lambda m: -np.nanmean([_r2(b, m, "raw") for b in DATASETS]))
_y_pos = np.arange(len(_order))
_bar_w = 0.27
for ax, bench in zip(axes, DATASETS):
    for _k, _red in enumerate(["raw", "pca", "isomap"]):
        _vals = [_r2(bench, m, _red) for m in _order]
        ax.barh(_y_pos + (1 - _k) * _bar_w, _vals, height=_bar_w,
                label=_red, edgecolor="white", linewidth=0.5,
                color={"raw": "0.2", "pca": "#4878A8", "isomap": "#63A363"}[_red])
    ax.set_yticks(_y_pos)
    ax.set_yticklabels(
        [f"{m}{' *' if m in LEGACY_METRICS else ''}" for m in _order], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xlabel("R² (pass@1 drop ~ distance from baseline)", fontsize=9)
    ax.set_title(BENCHMARK_LABELS[bench], fontsize=11, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)
_h, _l = axes[0].get_legend_handles_labels()
fig.legend(_h, _l, fontsize=9, ncol=3, loc="lower center",
           bbox_to_anchor=(0.5, -0.015), title="distance used", frameon=False)
fig.suptitle("Which distributional distance predicts real degradation?\n"
             "13 model variants per panel; * = the four metrics used elsewhere in "
             "this repository", fontsize=13, fontweight="bold", y=1.0)
fig.tight_layout()
_out = FIGURES_DIR / "metric_r2_comparison.png"
fig.savefig(_out, dpi=150, bbox_inches="tight")
print(f"Saved: {_out}")
plt.show()


### The regression itself, for the best and worst metric

Bar charts of R² hide whether a good score comes from a genuine linear trend or
from one leveraged point. These are the actual scatters.


In [ ]:
_ranked = sorted(METRIC_NAMES, key=lambda m: -np.nanmean([_r2(b, m, "raw") for b in DATASETS]))
_show = list(dict.fromkeys([_ranked[0], "jsd", _ranked[-1]]))

fig, axes = plt.subplots(len(_show), 3, figsize=(16, 4.6 * len(_show)))
for _r, _name in enumerate(_show):
    for _c, _bench in enumerate(DATASETS):
        ax = axes[_r, _c]
        _D = MATRICES[_bench][_name]
        _x = _D[0]
        _y = np.array([0.0 if m["cal"] == "baseline"
                       else DROP.get((m["cal"], _bench, int(m["level"])), np.nan)
                       for m in ROW_META], dtype=float)
        for _i, _meta in enumerate(ROW_META):
            _colour = "black" if _meta["cal"] == "baseline" else CAL_COLORS[_meta["cal"]]
            ax.scatter(_x[_i], _y[_i], c=_colour,
                       marker="*" if _meta["cal"] == "baseline" else "o",
                       s=260 if _meta["cal"] == "baseline" else 80,
                       edgecolors="k", linewidths=0.4, zorder=3)
        _ok = np.isfinite(_x) & np.isfinite(_y)
        if _ok.sum() >= 2:
            _fit = np.polyfit(_x[_ok], _y[_ok], 1)
            _xs = np.linspace(_x[_ok].min(), _x[_ok].max(), 50)
            ax.plot(_xs, np.polyval(_fit, _xs), "--", c="0.4", lw=1.2, zorder=2)
        ax.set_title(f"{_name} — {BENCHMARK_LABELS[_bench]}   "
                     f"R² = {_r2(_bench, _name, 'raw'):.3f}", fontsize=10)
        ax.set_xlabel(f"{_name} distance from baseline", fontsize=8)
        if _c == 0:
            ax.set_ylabel("pass@1 drop", fontsize=9)
        ax.grid(alpha=0.25)
fig.legend(handles=[mpatches.Patch(color=CAL_COLORS[c], label=f"cal = {BENCHMARK_LABELS[c]}")
                    for c in DATASETS]
           + [plt.scatter([], [], c="black", marker="*", s=150, label="baseline")],
           loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Best raw predictor, the incumbent (JSD), and the worst — "
             "the regressions behind the bars", fontsize=13, fontweight="bold", y=1.0)
fig.tight_layout()
_out = FIGURES_DIR / "metric_r2_scatter.png"
fig.savefig(_out, dpi=150, bbox_inches="tight")
print(f"Saved: {_out}")
plt.show()


## Dimension-reduced maps for every metric

The same picture notebooks 05 and 07 draw, now with the metric varying instead
of the reducer. One sheet per reducer: 16 rows (metrics) × 3 columns
(benchmarks). If the choice of distance were immaterial, every row would show
the same arrangement of points.


In [ ]:
def plot_metric_grid(reducer: str) -> Path:
    """16 metrics x 3 benchmarks in one sheet, for a single reducer."""
    fig, axes = plt.subplots(len(METRIC_NAMES), len(DATASETS),
                             figsize=(13, 2.8 * len(METRIC_NAMES)))
    for _r, name in enumerate(METRIC_NAMES):
        for _c, bench in enumerate(DATASETS):
            ax = axes[_r, _c]
            coords = EMBEDDINGS[(bench, name, reducer)]
            ax.scatter(coords[0, 0], coords[0, 1], c="black", marker="*", s=300, zorder=5)
            for _i, meta in enumerate(ROW_META[1:], start=1):
                ax.scatter(coords[_i, 0], coords[_i, 1], c=CAL_COLORS[meta["cal"]],
                           s=90, zorder=3, edgecolors="k", linewidths=0.3)
                ax.annotate(f"L{meta['level']}", (coords[_i, 0], coords[_i, 1]),
                            xytext=(4, 3), textcoords="offset points", fontsize=6)
            ax.set_xticks([])
            ax.set_yticks([])
            if _r == 0:
                ax.set_title(BENCHMARK_LABELS[bench], fontsize=10, fontweight="bold")
            if _c == 0:
                ax.set_ylabel(f"{name}\n({METRIC_INFO[name].family})",
                              fontsize=8, fontweight="bold")
    fig.legend(handles=[mpatches.Patch(color=CAL_COLORS[c], label=f"cal = {BENCHMARK_LABELS[c]}")
                        for c in DATASETS]
               + [plt.scatter([], [], c="black", marker="*", s=150, label="baseline (L0)")],
               loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, 0.0))
    fig.suptitle(f"{reducer.upper()} of the 13 model variants — "
                 "rows = distributional distance, cols = eval benchmark",
                 fontsize=13, fontweight="bold", y=1.0)
    fig.tight_layout(rect=(0, 0.012, 1, 0.995))
    out = FIGURES_DIR / f"metric_grid_{reducer}.png"
    fig.savefig(out, dpi=100, bbox_inches="tight")
    plt.close(fig)
    return out


for _red in REDUCERS_USED:
    print(f"Saved: {plot_metric_grid(_red)}")


In [ ]:
# Headline sheet: one representative metric per empirical family, side by side,
# on the benchmark with the most eval examples.
_HEADLINE = ["jsd", "kld", "chisq", "emd", "cosine", "chamfer"]
_BENCH = "gsm8k"

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, name in zip(axes.ravel(), _HEADLINE):
    coords = EMBEDDINGS[(_BENCH, name, "pca")]
    ax.scatter(coords[0, 0], coords[0, 1], c="black", marker="*", s=420, zorder=5)
    ax.annotate("base", (coords[0, 0], coords[0, 1]), xytext=(6, 6),
                textcoords="offset points", fontsize=9, fontweight="bold")
    for _i, meta in enumerate(ROW_META[1:], start=1):
        ax.scatter(coords[_i, 0], coords[_i, 1], c=CAL_COLORS[meta["cal"]], s=150,
                   zorder=3, edgecolors="k", linewidths=0.4)
        ax.annotate(f"L{meta['level']}", (coords[_i, 0], coords[_i, 1]),
                    xytext=(5, 4), textcoords="offset points", fontsize=7.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"{METRIC_INFO[name].label}  ({name})\n"
                 f"{METRIC_INFO[name].family} · R² = {_r2(_BENCH, name, 'raw'):.3f}",
                 fontsize=10, fontweight="bold")
fig.legend(handles=[mpatches.Patch(color=CAL_COLORS[c], label=f"cal = {BENCHMARK_LABELS[c]}")
                    for c in DATASETS]
           + [plt.scatter([], [], c="black", marker="*", s=150, label="baseline (L0)")],
           loc="lower center", ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.03))
fig.suptitle(f"Same 13 models, same reducer (PCA), six different notions of distance — "
             f"{BENCHMARK_LABELS[_BENCH]}", fontsize=14, fontweight="bold", y=1.0)
fig.tight_layout()
_out = FIGURES_DIR / "metric_headline_pca.png"
fig.savefig(_out, dpi=150, bbox_inches="tight")
print(f"Saved: {_out}")
plt.show()


## Verdict

**The choice of metric is not load-bearing.** Across 360 metric pairs the
lowest rank correlation anywhere is **ρ = 0.838**, the median is 0.968, and the
entire dendrogram fits under 1 − ρ = 0.064. Every one of the fifteen
alternatives is significantly related to JSD by a Mantel test (r ≥ 0.919,
p = 0.001 at 999 permutations). Nothing in notebooks 04, 05 or 07 turns on
having picked JSD; had any of the other fifteen been chosen instead, the same
model pairs would have come out close and the same ones far apart.

Two consequences worth spelling out, because they cut in opposite directions:

- *Reassuring.* The v1 conclusions are robust to the metric. This closes a
  criticism the original four never answered.
- *Deflationary.* Sixteen measures spanning four representations produce
  essentially one ordering, so none of them is telling you anything the others
  are not. Adding a seventeenth would be wasted effort. The signal being
  measured is coarse: at 80% pruning, models differ from the baseline in a way
  that every reasonable notion of distance picks up equally.

**On which one predicts degradation best.** EMD leads on mean R² (0.855) and
JSD is eighth (0.826) — but fifteen of the sixteen fall inside a 0.055 band on
13 data points, which is noise. The honest reading is that they are tied and
EMD's win should not be quoted as a result. Only `renyi2` separates (0.663),
and it does so by being *better* than everything else on HumanEval+ (0.805 vs
0.637 for JSD) while being much worse on ARC-Challenge — the signature of a
heavy-tailed measure fitting 33 tasks' worth of noise, not of a better metric.

**On the two that misbehave.** χ²'s scale is set by the ε guard rather than by
the data: a single position where the two models' top-5 sets are disjoint costs
it ~1e12, and about 8% of positions at 80% pruning are disjoint (sampled over 180
tasks per benchmark), so its matrix is close to a disjoint-position counter
times a large constant. That counter
happens to track damage well (R² 0.98 on ARC and GSM8K), so the *regression* is
fine — what is meaningless is the value's magnitude, and any reading of it as
"how much divergence". `renyi2` is the same quantity on a log scale and is the
measure to reach for if a tail-sensitive divergence is genuinely wanted.

**On Euclidean embeddability — the one place the metric choice does matter.**
`jsd`, `tv`, `hellinger`, `l2`, `emd`, `wasserstein2` and `chamfer` all have a
classical-MDS negative eigenvalue mass of **exactly zero** on all three
benchmarks: their 13-point distance matrices are Euclidean, and PCA or LLE on
them discards nothing. `kld` discards 20–31% and `chisq` 31–32%. So the KLD
panels in notebooks 05 and 07 are drawings of a materially different object
from the matrix that was handed in, and the JSD panels beside them are not.
That is a reason to prefer a bounded symmetric measure for anything that goes
through classical MDS, and it is the only recommendation in this notebook that
the data supports strongly.

**On what this does not establish.** Every number here is over 13 variants and
78 model pairs, on one model family. That is enough to compare metrics against
each other on the same data and not enough to make claims about pruning in
general; notebook 07's 232-variant analysis is where the statistical weight is.
This notebook also cannot say whether a metric that predicts `pass@1` drop
across pruning *levels* would separate calibration *sources* — the question
notebook 07's Mantel tests address, and the one where a metric difference could
still show up.


In [ ]:
print("Outputs")
print("=" * 62)
for _p in sorted(FIGURES_DIR.glob("*.png")):
    print(f"  {_p.relative_to(RESULTS_DIR)}  ({_p.stat().st_size / 1024:.0f} KB)")
for _n in ("metric_scale_audit.csv", "metric_mds_spectrum.csv",
           "metric_agreement.csv", "metric_family_r2.csv"):
    _p = RESULTS_DIR / _n
    if _p.exists():
        print(f"  {_n}  ({sum(1 for _ in _p.open()) - 1} rows)")
_new = [p for m in METRIC_NAMES if m not in LEGACY_METRICS
        for b in DATASETS if (p := _cache_path(b, m)).exists()]
print(f"  {len(_new)} new pairwise distance matrices cached in results/")
